In [ ]:
import kagglehub as khub
import warnings
warnings.filterwarnings('ignore')

In [ ]:
path = khub.dataset_download('aneerbansaha/sentiment')

Using Colab cache for faster access to the 'sentiment' dataset.


In [ ]:
path

'/kaggle/input/sentiment'

In [ ]:
import tensorflow as tf
import os

train_path = os.path.join(path, 'train')
val_path = os.path.join(path, 'val')

In [ ]:
image_height = 224
image_width = 224
batch_size = 64

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    subset = 'training',
    seed=123,
    validation_split=0.2)

Found 28709 files belonging to 7 classes.
Using 22968 files for training.


In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    subset = 'validation',
    seed=123,
    validation_split=0.2)

Found 28709 files belonging to 7 classes.
Using 5741 files for validation.


In [ ]:
original_class_name_for_display = train_ds.class_names
print(original_class_name_for_display)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [ ]:
def normalize(image, label):
    image = tf.cast(image/255.0, tf.float32)
    return image, label

In [ ]:
train_ds = train_ds.map(normalize)
val_ds = val_ds.map(normalize)

In [ ]:
def map_multiclass(image, label):
    label = tf.cast(label, tf.int32)
    return image, label

In [ ]:
train_ds = train_ds.map(map_multiclass)
val_ds = val_ds.map(map_multiclass)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomFlip('vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)
])

In [ ]:
def augment(image, label):
    return data_augmentation(image), label

In [ ]:
train_ds_augmented = train_ds.map(augment)

In [ ]:
# model = tf.keras.Sequential([

#     tf.keras.layers.Conv2D(32, (3, 3), input_shape=(image_height, image_width, 3)),
#     tf.keras.layers.BatchNormalization(),
#     tf.keras.layers.Activation('relu'),
#     tf.keras.layers.MaxPooling2D((2, 2)),

#     tf.keras.layers.Conv2D(64, (3, 3)),
#     tf.keras.layers.BatchNormalization(),
#     tf.keras.layers.Activation('relu'),
#     tf.keras.layers.MaxPooling2D((2, 2)),

#     tf.keras.layers.Conv2D(128, (3, 3)),
#     tf.keras.layers.BatchNormalization(),
#     tf.keras.layers.Activation('relu'),
#     tf.keras.layers.MaxPooling2D((2, 2)),

#     tf.keras.layers.Conv2D(256, (3, 3)),
#     tf.keras.layers.BatchNormalization(),
#     tf.keras.layers.Activation('relu'),
#     tf.keras.layers.MaxPooling2D((2, 2)),

#     tf.keras.layers.Flatten(),
#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dropout(0.5),

#     tf.keras.layers.Dense(256, activation='relu'),
#     tf.keras.layers.Dropout(0.5),

#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dropout(0.5),

#     tf.keras.layers.Dense(7, activation='softmax')
# ])

In [ ]:
model.summary()

Model: "functional_18"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_18      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_18[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,762,839 (10.54 MB)

 Trainable params: 167,431 (654.03 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

 Optimizer params: 334,864 (1.28 MB)

In [ ]:
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
# model.compile(
#     optimizer=optimizer,
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

In [ ]:
# Epochs = 10
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

# history = model.fit(train_ds_augmented,
#                     epochs=Epochs,
#                     validation_data=val_ds,
#                     callbacks=[early_stopping])

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Custom head
x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(7, activation='softmax')(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [180]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

history = model.fit(train_ds_augmented,
                    epochs=10,
                    validation_data=val_ds,
                    callbacks=[early_stopping])

Epoch 1/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 350s 948ms/step - accuracy: 0.2284 - loss: 2.4677 - val_accuracy: 0.3308 - val_loss: 1.7633
Epoch 2/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 304s 845ms/step - accuracy: 0.2974 - loss: 2.0258 - val_accuracy: 0.3508 - val_loss: 1.7230
Epoch 3/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 305s 849ms/step - accuracy: 0.3120 - loss: 1.9014 - val_accuracy: 0.3656 - val_loss: 1.6658
Epoch 4/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 306s 852ms/step - accuracy: 0.3262 - loss: 1.8191 - val_accuracy: 0.3830 - val_loss: 1.6135
Epoch 5/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 313s 870ms/step - accuracy: 0.3459 - loss: 1.7373 - val_accuracy: 0.3862 - val_loss: 1.5964
Epoch 6/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 305s 849ms/step - accuracy: 0.3476 - loss: 1.7048 - val_accuracy: 0.3830 - val_loss: 1.5764
Epoch 7/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 305s 848ms/step - accuracy: 0.3574 - loss: 1.6711 - val_accuracy: 0.3926 - val_loss: 1.5589
Epoch 8/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 307s 854ms/step - accuracy: 0.3686 -

In [184]:
base_model.trainable = True

# Freeze most layers, unfreeze top few
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 340s 896ms/step - accuracy: 0.3292 - loss: 1.7203 - val_accuracy: 0.3578 - val_loss: 2.1303
Epoch 2/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 305s 848ms/step - accuracy: 0.3590 - loss: 1.6623 - val_accuracy: 0.4005 - val_loss: 1.7242
Epoch 3/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 307s 855ms/step - accuracy: 0.3754 - loss: 1.6224 - val_accuracy: 0.4104 - val_loss: 1.6135
Epoch 4/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 305s 848ms/step - accuracy: 0.3828 - loss: 1.5942 - val_accuracy: 0.4388 - val_loss: 1.4949
Epoch 5/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 305s 850ms/step - accuracy: 0.3952 - loss: 1.5664 - val_accuracy: 0.4463 - val_loss: 1.4584
Epoch 6/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 304s 846ms/step - accuracy: 0.4067 - loss: 1.5439 - val_accuracy: 0.4604 - val_loss: 1.4175
Epoch 7/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 310s 864ms/step - accuracy: 0.4141 - loss: 1.5212 - val_accuracy: 0.4680 - val_loss: 1.3896
Epoch 8/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 304s 847ms/step - accuracy: 0.4231 -

To potentially improve accuracy further, let's try unfreezing more layers of the `MobileNetV2` base model for more extensive fine-tuning. We will recompile the model with a very small learning rate and train for more epochs.

In [185]:
# Unfreeze the base model again
base_model.trainable = True

# Unfreeze even more layers for further fine-tuning (e.g., last 60 layers)
# You can experiment with this number
for layer in base_model.layers[:-60]:
    layer.trainable = False

# Recompile the model with a very low learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6), # Even smaller learning rate
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Number of trainable variables after unfreezing more layers:", len(model.trainable_variables))

# Continue training for more epochs
history_fine_tuned = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=20, # Increased epochs for more fine-tuning
    callbacks=[early_stopping] # Re-using early stopping
)

Number of trainable variables after unfreezing more layers: 38
Epoch 1/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 341s 899ms/step - accuracy: 0.4394 - loss: 1.4610 - val_accuracy: 0.4898 - val_loss: 1.3365
Epoch 2/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 360s 859ms/step - accuracy: 0.4432 - loss: 1.4558 - val_accuracy: 0.4907 - val_loss: 1.3349
Epoch 3/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 308s 856ms/step - accuracy: 0.4501 - loss: 1.4416 - val_accuracy: 0.4959 - val_loss: 1.3239
Epoch 4/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 308s 859ms/step - accuracy: 0.4510 - loss: 1.4328 - val_accuracy: 0.4999 - val_loss: 1.3184
Epoch 5/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 310s 863ms/step - accuracy: 0.4506 - loss: 1.4334 - val_accuracy: 0.4982 - val_loss: 1.3139
Epoch 6/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 315s 878ms/step - accuracy: 0.4512 - loss: 1.4231 - val_accuracy: 0.5043 - val_loss: 1.3091
Epoch 7/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 315s 875ms/step - accuracy: 0.4547 - loss: 1.4192 - val_accuracy: 0.5041 - val_loss: 1.3052
Epoch 8/20
35

KeyboardInterrupt: 

In [186]:
model_save_path = "./my_sentiment_model_epoch8.h5"
model.save(model_save_path)
print(f"Model saved to: {model_save_path}")

Model saved to: ./my_sentiment_model_epoch8.h5


In [187]:
model_save_path = "./my_sentiment_model_epoch8.keras"
model.save(model_save_path)
print(f"Model saved to: {model_save_path}")

Model saved to: ./my_sentiment_model_epoch8.keras


First, let's load and preprocess the test dataset. We'll assume the test data is located in a `test` subdirectory within the `path` variable.

In [191]:
import os
import tensorflow as tf

# Define the path to the test data
test_path = os.path.join(path, 'test')

# Load the test dataset
initial_test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    shuffle=False # No need to shuffle test data
)

# Capture class names before applying map functions
test_class_names = initial_test_ds.class_names

# Apply the same normalization and multiclass mapping functions
test_ds = initial_test_ds.map(normalize)
test_ds = test_ds.map(map_multiclass)

print(f"Found {len(test_class_names)} classes in the test set.")

Found 7178 files belonging to 7 classes.
Found 7 classes in the test set.


Now, let's load your saved model and evaluate it on this test data.

In [192]:
# Load the saved model
# Assuming the model was saved as 'my_sentiment_model_epoch8.keras'
loaded_model = tf.keras.models.load_model('./my_sentiment_model_epoch8.keras')

print("Model loaded successfully!")

# Evaluate the loaded model on the test dataset
loss, accuracy = loaded_model.evaluate(test_ds)

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Model loaded successfully!
113/113 ━━━━━━━━━━━━━━━━━━━━ 16s 85ms/step - accuracy: 0.5021 - loss: 1.3127

Test Loss: 1.3127
Test Accuracy: 0.5021
